In [8]:
import pandas as pd

df = pd.read_csv(r'../dataset_with_labels.csv')
df.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,2plbrEY59IikOBgBGLjaoe,Die With A Smile,"Lady Gaga, Bruno Mars",1,1,0,NaN,2025-02-17,98,False,...,-7.777,0,0.0304,0.3080,0.0000,0.122,0.535,157.969,3,Lower
1,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",2,1,4,NaN,2025-02-17,90,False,...,-7.546,1,0.1250,0.2510,0.0000,0.248,0.576,138.008,4,About_Average
2,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,Kendrick Lamar,3,-2,8,NaN,2025-02-17,92,True,...,-7.001,1,0.0776,0.0107,0.0000,0.141,0.214,101.061,4,Higher
3,4wJ5Qq0jBN4ajy7ouZIV1c,APT.,"ROSÉ, Bruno Mars",4,0,-2,NaN,2025-02-17,89,False,...,-4.477,0,0.2600,0.0283,0.0000,0.355,0.939,149.027,4,Higher
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,-2,NaN,2025-02-17,96,False,...,-10.171,1,0.0358,0.2000,0.0608,0.117,0.438,104.978,4,About_Average


In [9]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'is_explicit',
       'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    RandomForestRegressor(n_estimators=200, min_samples_split=2, min_samples_leaf=2, max_features=0.8, max_depth=30, n_jobs=4)
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)




KeyboardInterrupt: 

In [18]:
print("Random Forests Regression")
print("Mean Absolute Error: ", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error: ", mean_squared_error(y_test, y_pred))
print("Root Mean Squared Error: ", root_mean_squared_error(y_test, y_pred))
print("R2 Score: ", r2_score(y_test, y_pred))

for i in range(10):
    print(f"Predicted: {y_pred[i]}, Actual: {y_test.iloc[i]}")


Random Forests Regression
Mean Absolute Error:  2.1435923264793133
Mean Squared Error:  24.21961990462585
Root Mean Squared Error:  4.921343302862121
R2 Score:  0.9025006862877021
Predicted: 85.27227989791896, Actual: 82
Predicted: 83.650341260041, Actual: 85
Predicted: 81.81910832913809, Actual: 85
Predicted: 60.482230484069554, Actual: 61
Predicted: 97.98386630363814, Actual: 97
Predicted: 59.13620531190431, Actual: 60
Predicted: 91.75887421737815, Actual: 92
Predicted: 82.151861476899, Actual: 79
Predicted: 89.97971550768644, Actual: 87
Predicted: 84.34539238236738, Actual: 90


param_distributions = {
    'randomforestregressor__n_estimators': [50, 100, 200, 300],
    'randomforestregressor__max_depth': [5, 10, 20, 30, None],
    'randomforestregressor__min_samples_split': [2, 5, 10],
    'randomforestregressor__min_samples_leaf': [1, 2, 4],
    'randomforestregressor__max_features': ['sqrt', 'log2', 0.8]
}

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold

RFRGrid = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_iter=20,
    random_state=42,
    verbose=1
)

RFRGrid.fit(X_train, y_train)
print("Random Forests Regression with Grid Search")
print(RFRGrid.best_params_)
print(RFRGrid.best_score_)

Default = 25.49200082846408
{'randomforestregressor__n_estimators': 200, 'randomforestregressor__max_depth': 30} = -27.077155763516515
{'randomforestregressor__n_estimators': 200, 'randomforestregressor__min_samples_split': 2, 'randomforestregressor__min_samples_leaf': 2, 'randomforestregressor__max_features': 0.8, 'randomforestregressor__max_depth': 30}
-25.36150368665468, 1372 minutes


